# Baca data EEG Muse dari folder Google Drive
Notebook ini mengunduh semua file dari folder Google Drive publik, lalu mencoba membaca file EEG (CSV/TXT/EDF/FIF/XDF).

In [ ]:
# Jalankan sekali di Google Colab
!pip -q install gdown pandas mne pyxdf

In [ ]:
from pathlib import Path
import re
import pandas as pd

try:
    import gdown
except ImportError:
    raise RuntimeError('Package gdown belum terpasang. Jalankan cell instalasi di atas.')

In [ ]:
# Link folder yang Anda berikan
drive_folder_url = 'https://drive.google.com/drive/folders/1vkm6dITmXVIVuZNobfbRcyKTQH13qwI-?usp=drive_link'

m = re.search(r'/folders/([a-zA-Z0-9_-]+)', drive_folder_url)
if not m:
    raise ValueError('Folder ID tidak ditemukan dari URL.')
folder_id = m.group(1)
folder_id

In [ ]:
# Unduh seluruh isi folder publik ke lokal runtime (mis. /content/muse_data di Colab)
output_dir = Path('muse_data')
output_dir.mkdir(parents=True, exist_ok=True)

downloaded = gdown.download_folder(id=folder_id, output=str(output_dir), quiet=False, use_cookies=False)
print('Jumlah file terunduh:', 0 if downloaded is None else len(downloaded))

In [ ]:
# Lihat semua file yang ada
all_files = sorted([p for p in output_dir.rglob('*') if p.is_file()])
for p in all_files:
    print(p)

In [ ]:
# Cari file EEG umum
candidate_ext = {'.csv', '.txt', '.edf', '.fif', '.xdf'}
eeg_files = [p for p in all_files if p.suffix.lower() in candidate_ext]

if not eeg_files:
    raise FileNotFoundError('Tidak menemukan file EEG dengan ekstensi umum: ' + ', '.join(sorted(candidate_ext)))

print('Kandidat file EEG:')
for f in eeg_files:
    print('-', f)

In [ ]:
# Contoh: baca file pertama
target = eeg_files[0]
print('Membaca:', target)

ext = target.suffix.lower()

if ext in {'.csv', '.txt'}:
    # Umumnya ekspor Muse (mis. Mind Monitor) berbentuk CSV
    df = pd.read_csv(target)
    display(df.head())
    print('Shape:', df.shape)

elif ext == '.edf':
    import mne
    raw = mne.io.read_raw_edf(str(target), preload=False)
    print(raw)
    print(raw.ch_names)

elif ext == '.fif':
    import mne
    raw = mne.io.read_raw_fif(str(target), preload=False)
    print(raw)
    print(raw.ch_names)

elif ext == '.xdf':
    import pyxdf
    streams, header = pyxdf.load_xdf(str(target))
    print('Jumlah stream:', len(streams))
    for i, s in enumerate(streams):
        name = s.get('info', {}).get('name', ['unknown'])[0]
        print(f'Stream {i}:', name, '| samples =', len(s.get('time_stamps', [])))

else:
    raise ValueError(f'Format {ext} belum ditangani di contoh ini.')